In [2]:
import math
import random
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import nltk

nltk.download("gutenberg", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import gutenberg

TRAIN_FILES = ["shakespeare-caesar.txt", "shakespeare-macbeth.txt"]
TEST_FILES = ["shakespeare-hamlet.txt"]

BOS, EOS, UNK = "<s>", "</s>", "<UNK>"

## 1. Data preprocessing
- Sentence-split + word-tokenize (NLTK's Gutenberg reader already gives tokenized sentences)
- Lowercase
- Remove punctuation / special characters (keep only tokens that contain letters or digits,
  and strip any non-alphanumeric characters left inside a token)
- Stopwords are KEPT (a language model must model function words as well)

In [16]:
def preprocess(file_txts):
    """Return a list of sentences; each sentence is a list of clean lowercase words."""
    sentences = []
    for fid in file_txts:
        for sent in gutenberg.sents(fid):
            words = []
            for tok in sent:
                tok = tok.lower()
                tok = re.sub(r"[^a-z0-9]", "", tok)   # drop punctuation / special chars
                if tok:
                    words.append(tok)
            if words:
                sentences.append(words)
    return sentences


train_sents = preprocess(TRAIN_FILES)
test_sents = preprocess(TEST_FILES)

n_train_tokens = sum(len(s) for s in train_sents)
n_test_tokens = sum(len(s) for s in test_sents)
print(f"Train: {len(train_sents)} sentences, {n_train_tokens} tokens")
print(f"Test : {len(test_sents)} sentences, {n_test_tokens} tokens")
print("Example:", train_sents[10])

Train: 4070 sentences, 39224 tokens
Test : 3106 sentences, 30271 tokens
Example: ['mur']


## 2. The N-gram model
P(w_i | w_{i-n+1..i-1}) = (C(context, w) + k) / (C(context) + k*V)   (Laplace: k = 1)

* Each sentence is padded with (n-1) `<s>` tokens and one `</s>`.
* The vocabulary is built from the training data; at test time unseen words become `<UNK>`.
* V = |train vocab| + `<UNK>` + `</s>`  (`<s>` is never predicted so it is not in V).

In [11]:
class NGramLM:
    def __init__(self, n=2, k=1.0):
        self.n = n
        self.k = k                                   # k = 1 -> Laplace smoothing
        self.ngram_counts = Counter()                # C(context + word)
        self.context_counts = Counter()              # C(context)
        self.next_words = defaultdict(Counter)       # context -> Counter(next word), used for generation
        self.vocab = set()

    # ---------- helpers ----------
    def _pad(self, sent):
        return [BOS] * (self.n - 1) + sent + [EOS]

    def _map_unk(self, sent):
        return [w if w in self.vocab else UNK for w in sent]

    # ---------- training ----------
    def fit(self, sentences, min_count=1):
        """min_count=1 keeps every training word; min_count=2 turns words seen once into <UNK>."""
        word_counts = Counter(w for s in sentences for w in s)
        self.vocab = {w for w, c in word_counts.items() if c >= min_count} | {EOS, UNK}
        self.V = len(self.vocab)
        sentences = [self._map_unk(s) for s in sentences]
        for sent in sentences:
            padded = self._pad(sent)
            for i in range(self.n - 1, len(padded)):
                ctx = tuple(padded[i - self.n + 1:i])
                w = padded[i]
                self.ngram_counts[ctx + (w,)] += 1
                self.context_counts[ctx] += 1
                self.next_words[ctx][w] += 1
        return self

    # ---------- probability ----------
    def prob(self, word, ctx):
        """Smoothed probability of `word` given the context tuple (length n-1)."""
        num = self.ngram_counts.get(ctx + (word,), 0) + self.k
        den = self.context_counts.get(ctx, 0) + self.k * self.V
        return num / den

    # ---------- evaluation ----------
    def perplexity(self, sentences):
        log_sum, N = 0.0, 0
        for sent in sentences:
            padded = self._pad(self._map_unk(sent))
            for i in range(self.n - 1, len(padded)):
                ctx = tuple(padded[i - self.n + 1:i])
                log_sum += math.log(self.prob(padded[i], ctx))
                N += 1
        return math.exp(-log_sum / N)

    # ---------- generation ----------
    def generate(self, max_words=30, seed=None, smoothed=False):
        """
        Sample a passage of `max_words` words, word by word.
        smoothed=False: sample from the observed continuations of the context
                        (backing off to a shorter context if this one was never seen).
        smoothed=True : sample from the full Laplace-smoothed distribution over V.
        When </s> is sampled the sentence ends ('.') and a new one starts from <s> padding.
        """
        rng = random.Random(seed)
        start = [BOS] * (self.n - 1)
        out, produced = list(start), 0
        while produced < max_words:
            ctx = tuple(out[len(out) - (self.n - 1):]) if self.n > 1 else ()
            if smoothed:
                words = list(self.vocab)
                weights = [self.prob(w, ctx) for w in words]
                nxt = rng.choices(words, weights=weights)[0]
            else:
                c = ctx
                while c not in self.next_words and len(c) > 0:   # back-off
                    c = c[1:]
                cand = self.next_words[c]
                nxt = rng.choices(list(cand.keys()), weights=list(cand.values()))[0]
            if nxt == EOS:
                out.append(".")
                out.extend(start)          # new sentence
                continue
            out.append(nxt)
            produced += 1
        return " ".join(w for w in out if w != BOS)

## 3. Step 1 — Unigram model (baseline)
A unigram model has no context: P(w) = C(w) / total_tokens (with Laplace smoothing).

In [12]:
unigram = NGramLM(n=1, k=1.0).fit(train_sents)

print(f"Vocabulary size : {unigram.V}")
print(f"Unigram perplexity (train): {unigram.perplexity(train_sents):.2f}")
print(f"Unigram perplexity (test) : {unigram.perplexity(test_sents):.2f}")

Vocabulary size : 5065
Unigram perplexity (train): 525.23
Unigram perplexity (test) : 715.79


## 3. Step 2 — Bigram, Trigram, and chosen N-gram models
Each model conditions on the previous (n-1) words.

In [ ]:
bigram  = NGramLM(n=2, k=1.0).fit(train_sents)
trigram = NGramLM(n=3, k=1.0).fit(train_sents)
fourgram = NGramLM(n=4, k=1.0).fit(train_sents)
#print(f"Vocabulary size : {trigram.V}")
for name, model in [("Bigram", bigram), ("Trigram", trigram), ("4-gram", fourgram)]:
    print(f"{name}  -> train perplexity: {model.perplexity(train_sents):.2f}  "
          f"| test perplexity: {model.perplexity(test_sents):.2f}")

Vocabulary size : 5065
Bigram  -> train perplexity: 1201.66  | test perplexity: 2421.43
Trigram  -> train perplexity: 1797.27  | test perplexity: 4053.36
4-gram  -> train perplexity: 1866.21  | test perplexity: 4256.27


## 3. Step 3 — Laplace smoothing demonstration
Laplace smoothing adds k=1 to every count so unseen n-grams get a small non-zero probability instead of 0.

Formula: P(w | ctx) = (C(ctx, w) + k) / (C(ctx) + k * V)

In [14]:
# Show the effect of smoothing on an unseen bigram
unseen_word = "hamlet"   # from test set, not in Caesar/Macbeth training
ctx = ("the",)

raw_count   = bigram.ngram_counts.get(ctx + (unseen_word,), 0)
smoothed_p  = bigram.prob(unseen_word, ctx)
seen_p      = bigram.prob("king", ctx)

print(f"Raw count of ('the', '{unseen_word}')  : {raw_count}")
print(f"Smoothed P('{unseen_word}' | 'the')    : {smoothed_p:.6f}  <- non-zero thanks to Laplace")
print(f"Smoothed P('king'        | 'the')    : {seen_p:.6f}")

Raw count of ('the', 'hamlet')  : 0
Smoothed P('hamlet' | 'the')    : 0.000159  <- non-zero thanks to Laplace
Smoothed P('king'        | 'the')    : 0.002224


## 3. Step 4 — Text generation
Generate sample text using each model to see how more context produces more coherent output.

In [15]:
SEED = 42

for name, model in [("Unigram", unigram), ("Bigram", bigram), ("Trigram", trigram), ("4-gram", fourgram)]:
    print(f"--- {name} ---")
    print(model.generate(max_words=30, seed=SEED))
    print()

--- Unigram ---
bru the on what crowne did pindarus . haue of a her the is himselfe so a stand none the within god that and seas me . . wrong shall suffers giuen will

--- Bigram ---
cai . i haue such another more then countrymen and tell thee a blunt man drop of marriage tell me the dearest partner you gon and i may not gone master

--- Trigram ---
cai . i do marke antony by our permission else shall you not made an vniuersall shout that tyber trembled vnderneath her bankes to heare and since you know not how

--- 4-gram ---
cai . i do not like your faults . enter brutus lucillius and the army . calp . scena secunda . 1 . and euen now to crown my thoughts with acts be it thoght done

